[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/erickcmendes/CEUB/blob/main/Otimiza%C3%A7%C3%A3o%20para%20Ci%C3%AAncia%20de%20Dados/crypto-portfolio-optimization/notebooks/a2_colab_pt.ipynb)

# Otimização de Carteiras de Criptomoedas sob Restrições de Cardinalidade
## Avaliação 2 (A2) · Etapas 4, 5 e 6 · Otimização Aplicada à Ciência de Dados
**Instituição:** Centro Universitário de Brasília (CEUB)
**Discente:** Erick Mendes
**Entrega final:** 07/12/2026

Este notebook continua a A1. Na A1 formulamos e resolvemos o problema de seleção de carteira com limite de cardinalidade
(no máximo $K$ ativos, pesos entre 5% e 40%) por métodos exatos (Pyomo com CBC, Ipopt e Couenne) e por uma metaheurística
(Algoritmo Genético com DEAP). Aqui:

| Etapa | Pergunta | Seção |
|---|---|---|
| **4 · Aplicação prática e ML** | O modelo otimizado funciona em um pipeline real de dados, fora da amostra? Previsões de ML melhoram a decisão? | 4 |
| **5 · Análise comparativa** | Exato x metaheurística: tempo, CPU/RAM, convergência, exploração x explotação e sensibilidade | 5 |
| **6 · Visualização** | Painel interativo com as métricas e objetivos do projeto | 6 |

**Rastreabilidade:** `FR-07` (ML alimenta o otimizador), `FR-08` (comparação exato x metaheurística), `FR-09` (painel),
`FR-05`/`NFR-01` (execução única e reprodutível, sementes fixas). Toda a lógica está no pacote `portfolio_opt`;
o notebook apenas chama e explica.

**Como executar:** *Ambiente de execução → Executar tudo*. Tempo estimado no Colab: 6 a 10 minutos
(instalação dos solvers ≈ 1 min; o restante são os experimentos). Os tempos de execução medidos variam com a máquina;
todos os demais números (objetivos, pesos, métricas) são determinísticos.

---
## 1. Ambiente e solvers (`NFR-02`, `D-003`)
No Colab, instala os binários COIN-OR (CBC, Ipopt, Couenne) via extensões do IDAES e o pacote `portfolio_opt`
direto do repositório (na versão `v0.2.0`, se a tag existir). Localmente, usa o código em `../src`.

In [ ]:
# FR-05, NFR-02: instalação automática e configuração do ambiente
import os, sys, subprocess, shutil
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
REPO_URL = "https://github.com/erickcmendes/CEUB.git"
PROJECT_SUBDIR = "Otimização para Ciência de Dados/crypto-portfolio-optimization"
VERSION_TAG = "v0.2.0"

def _add_solver_dirs_to_path():
    candidates = [Path.home() / ".idaes" / "bin"]
    if os.environ.get("LOCALAPPDATA"):
        candidates.append(Path(os.environ["LOCALAPPDATA"]) / "idaes" / "bin")
    try:
        out = subprocess.run(["idaes", "bin-directory"], capture_output=True, text=True, timeout=60)
        if out.returncode == 0 and out.stdout.strip():
            candidates.insert(0, Path(out.stdout.strip().splitlines()[-1]))
    except Exception:
        pass
    for d in candidates:
        if d.exists() and str(d) not in os.environ["PATH"]:
            os.environ["PATH"] = str(d) + os.pathsep + os.environ["PATH"]

if IS_COLAB:
    print("Google Colab detectado: instalando dependências e solvers...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyomo", "idaes-pse", "deap", "gdown"], check=True)
    subprocess.run(["idaes", "get-extensions"], check=True, capture_output=True)
    if not Path("CEUB").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    tags = subprocess.run(["git", "-C", "CEUB", "tag"], capture_output=True, text=True).stdout.split()
    if VERSION_TAG in tags:
        subprocess.run(["git", "-C", "CEUB", "checkout", "-q", VERSION_TAG], check=True)
        print(f"Repositório fixado na versão {VERSION_TAG}.")
    project_dir = Path("CEUB") / PROJECT_SUBDIR
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(project_dir)], check=True)
    sys.path.insert(0, str((project_dir / "src").resolve()))
    os.chdir(project_dir)
else:
    root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.insert(0, str((root / "src").resolve()))
    os.chdir(root)

_add_solver_dirs_to_path()

import portfolio_opt
from portfolio_opt.solvers import check_solver_available
print(f"portfolio_opt {portfolio_opt.__version__} · Python {sys.version.split()[0]}")
for s in ["cbc", "ipopt", "couenne"]:
    print(f"  solver {s:8s}: {'disponível' if check_solver_available(s) else 'NÃO encontrado'}")
assert all(check_solver_available(s) for s in ["cbc", "ipopt", "couenne"]), \
    "Solvers ausentes: execute novamente esta célula (a instalação do IDAES pode falhar por rede)."

---
## 2. Parâmetros centrais (`FR-06`, `NFR-01`)
Os parâmetros do modelo são os mesmos da A1. Os parâmetros novos da A2 ficam todos aqui.

In [ ]:
# FR-06, NFR-01: célula única de parâmetros
from dataclasses import replace
import warnings, logging
import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display, HTML

from portfolio_opt.params import OptimizationParams

warnings.filterwarnings("ignore")
logging.getLogger("pyomo").setLevel(logging.ERROR)
pio.renderers.default = "colab"  # funciona no Colab e preserva os gráficos nas saídas salvas
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Modelo da A1 (mesmos valores)
PARAMS = OptimizationParams(
    cardinality_limit=5, min_weight=0.05, max_weight=0.40, target_daily_return=0.0010,
    cvar_alpha=0.95, risk_aversion_lambda=1.0,
    ga_seed=42, ga_pop_size=100, ga_generations=80, ga_cx_prob=0.80, ga_mut_prob=0.20,
    ga_tournament_size=3, ga_penalty_weight=1000.0, ga_mut_sigma=0.20, ga_mut_indpb=0.25,
)

# Etapa 4: pipeline fora da amostra
HORIZONTE_DIAS = 30        # rebalanceamento e horizonte da previsão
JANELA_TREINO_DIAS = 365   # janela móvel para covariância, média histórica e treino do ML
CUSTO_TRANSACAO = 0.001    # 0,1% sobre o giro da carteira
RIDGE_ALPHA = 10.0         # regularização L2 do modelo de previsão
ALPHAS_ROBUSTEZ = [1.0, 100.0]
# No backtest a meta de retorno é desligada: em meses de baixa ela tornaria o problema inviável.
# AG menor para manter o tempo (37 rebalanceamentos x 3 otimizações).
BACKTEST_PARAMS = replace(PARAMS, target_daily_return=-1.0, ga_pop_size=60, ga_generations=40)

# Etapa 5: comparação
SEMENTES_AG = list(range(10))
LIMITE_TEMPO_S = 120
LIMITE_TEMPO_SENSIBILIDADE_S = 60
K_VALORES = [2, 3, 4, 5, 7, 10]
LAMBDA_VALORES = [0.25, 0.5, 1.0, 2.0, 4.0]
N_VALORES = [5, 10, 15, 19]
SIGMAS_MUTACAO = [0.05, 0.20, 0.60]
TAMANHOS_TORNEIO = [2, 5, 10]
SEMENTES_GRADE = list(range(5))
print("Parâmetros carregados.")

---
## 3. Dados e integridade (`FR-05`, `NFR-05`, `D-004`)
Mesma base da A1: preços diários de 19 criptoativos líquidos, 2021 a 2024, conferidos por SHA-256.

In [ ]:
# FR-05, NFR-05, D-004: ingestão e validação
import urllib.request
from portfolio_opt.data import compute_file_sha256, load_and_validate_prices, compute_returns_and_scenarios

data_path = Path("data/crypto_prices.csv")
if not data_path.exists():
    data_path.parent.mkdir(exist_ok=True)
    raw = ("https://raw.githubusercontent.com/erickcmendes/CEUB/main/Otimiza%C3%A7%C3%A3o%20para%20Ci%C3%AAncia"
           "%20de%20Dados/crypto-portfolio-optimization/data/crypto_prices.csv")
    print("Baixando a base do GitHub...")
    urllib.request.urlretrieve(raw, data_path)

sha = compute_file_sha256(data_path)
print("SHA-256 confere com o snapshot oficial." if sha == PARAMS.expected_sha256 else f"Atenção: SHA-256 diferente ({sha}).")
df_prices = load_and_validate_prices(data_path, PARAMS.expected_sha256)
ret_matrix, mean_returns, cov_matrix = compute_returns_and_scenarios(df_prices)
print(f"{ret_matrix.shape[1]} ativos · {ret_matrix.shape[0]} dias de retorno "
      f"({ret_matrix.index.min():%d/%m/%Y} a {ret_matrix.index.max():%d/%m/%Y})")

---
## 4. Etapa 4 · O modelo otimizado em um pipeline real de dados (`FR-07`)

### 4.1 Por que validar fora da amostra
Na A1 o otimizador recebeu $\mu$ (retorno esperado) e $\Sigma$ (covariância) estimados com **os mesmos 4 anos** em que a carteira
foi avaliada. A carteira ótima da A1 (DOGE, SOL, BNB, AVAX, XRP) "promete" cerca de 179% ao ano, mas essa promessa só vale
se $\mu$ estiver certo. A pergunta operacional é: **se eu tivesse usado este modelo para decidir, mês a mês, com os dados
disponíveis em cada data, o que teria acontecido?**

### 4.2 Desenho do pipeline (walk-forward, sem vazamento de informação)
Em cada data de rebalanceamento $t$ (a cada 30 dias, de jan/2022 a dez/2024, 37 decisões):

1. **Estimação de risco:** $\Sigma_t$ com os últimos 365 dias até $t$.
2. **Estimação de retorno**, três alternativas que alimentam **o mesmo otimizador** (AG, $K=5$, pesos de 5% a 40%):
   - **ML (Ridge):** um modelo de regressão em painel (todos os ativos juntos) prevê o retorno médio diário dos próximos 30 dias.
     *Features* conhecidas em $t$: momentos de 7, 30 e 90 dias, volatilidades de 30 e 90 dias, momento relativo ao mercado e
     posto do momento entre os ativos. O treino usa apenas alvos **já realizados** em $t$ (linhas até $t-30$), com padronização e
     regularização L2. Previsões fora do intervalo visto no treino são cortadas (mecanismo de falha de `docs/ai-features.md`).
   - **Média histórica:** a abordagem da A1, média dos últimos 365 dias.
   - **Controle de mínima variância ($\mu = 0$):** o otimizador recebe só o risco. Serve para isolar o valor da previsão de retorno.
3. **Execução:** a carteira é mantida até a próxima data (os pesos flutuam com os preços), com custo de 0,1% sobre o giro.
4. **Referências passivas:** igual peso entre os 19 ativos (1/N) e BTC comprado e mantido.

Além do desempenho **operacional** (financeiro), medimos o desempenho **preditivo** das previsões contra o retorno que de fato
ocorreu nos 30 dias seguintes: RMSE, MAE, acerto de direção e o IC (correlação de Spearman entre previsão e realizado,
a métrica usual para ordenar ativos).

In [ ]:
# FR-07: backtest walk-forward (≈ 1 a 2 minutos)
from portfolio_opt.backtest import run_walk_forward_backtest, STRATEGIES

wf = run_walk_forward_backtest(
    ret_matrix, BACKTEST_PARAMS, horizon=HORIZONTE_DIAS, train_window=JANELA_TREINO_DIAS,
    cost_rate=CUSTO_TRANSACAO, ridge_alpha=RIDGE_ALPHA,
)
print(f"{len(wf.rebalance_dates)} rebalanceamentos: {wf.rebalance_dates[0]:%d/%m/%Y} a {wf.rebalance_dates[-1]:%d/%m/%Y}")

fmt_pct = ["Retorno Anual (CAGR)", "Volatilidade Anual", "Máx. Drawdown", "CVaR 95% Diário"]
display(wf.performance.style.format({**{c: "{:.1%}" for c in fmt_pct}, "Sharpe": "{:.2f}",
        "Giro Médio por Rebalanceamento": "{:.2f}", "Riqueza Final (R$1 inicial)": "{:.2f}"})
        .set_caption("Desempenho operacional fora da amostra"))
display(wf.forecast_quality.style.format("{:.4f}").set_caption("Desempenho preditivo das estimativas de retorno"))

In [ ]:
# FR-07, FR-09: visualizações da Etapa 4
from portfolio_opt.dashboard import fig_wealth, fig_drawdown, fig_monthly_ic, fig_allocation
for f in [fig_wealth(wf), fig_drawdown(wf), fig_monthly_ic(wf), fig_allocation(wf, STRATEGIES[0])]:
    f.show()

anual = wf.daily_returns.groupby(wf.daily_returns.index.year).apply(lambda d: (1 + d).prod() - 1)
display(anual.T.style.format("{:.1%}").set_caption("Retorno por ano civil"))

### 4.3 Robustez: o resultado depende da regularização do modelo?
Repetimos o pipeline com regularização mais fraca e mais forte. Se a conclusão mudar com o hiperparâmetro, ela não é confiável.

In [ ]:
# FR-07: sensibilidade ao hiperparâmetro do Ridge (≈ 2 a 4 minutos)
linhas = []
for a in ALPHAS_ROBUSTEZ + [RIDGE_ALPHA]:
    r = wf if a == RIDGE_ALPHA else run_walk_forward_backtest(
        ret_matrix, BACKTEST_PARAMS, horizon=HORIZONTE_DIAS, train_window=JANELA_TREINO_DIAS,
        cost_rate=CUSTO_TRANSACAO, ridge_alpha=a)
    linhas.append({"alpha do Ridge": a,
                   "Sharpe ML": r.performance.loc[STRATEGIES[0], "Sharpe"],
                   "Sharpe Média Histórica": r.performance.loc[STRATEGIES[1], "Sharpe"],
                   "Sharpe Mínima Variância": r.performance.loc[STRATEGIES[2], "Sharpe"],
                   "IC médio ML": r.forecast_quality.loc["Previsão ML (Ridge)", "IC de Spearman médio"]})
robustez = pd.DataFrame(linhas).sort_values("alpha do Ridge").set_index("alpha do Ridge")
display(robustez.style.format("{:.3f}"))

In [ ]:
# Resumo automático desta execução (os números abaixo vêm do que acabou de rodar)
p, q = wf.performance, wf.forecast_quality
ml, hi, mv = STRATEGIES[0], STRATEGIES[1], STRATEGIES[2]
print(f"Sharpe fora da amostra: ML {p.loc[ml,'Sharpe']:.2f} | média histórica {p.loc[hi,'Sharpe']:.2f} | "
      f"mínima variância {p.loc[mv,'Sharpe']:.2f} | 1/N {p.loc[STRATEGIES[3],'Sharpe']:.2f} | BTC {p.loc[STRATEGIES[4],'Sharpe']:.2f}")
print(f"Riqueza final de R$ 1: ML {p.loc[ml,'Riqueza Final (R$1 inicial)']:.2f} | média histórica "
      f"{p.loc[hi,'Riqueza Final (R$1 inicial)']:.2f} | mínima variância {p.loc[mv,'Riqueza Final (R$1 inicial)']:.2f}")
print(f"IC médio: ML {q.iloc[0]['IC de Spearman médio']:+.3f} | média histórica {q.iloc[1]['IC de Spearman médio']:+.3f}; "
      f"acerto de direção: ML {q.iloc[0]['Acerto de Direção (%)']:.1f}% | média histórica {q.iloc[1]['Acerto de Direção (%)']:.1f}%")

### 4.4 Análise da Etapa 4
*Números da execução de referência; a célula acima reproduz os da sua execução.*

**1. O ML melhorou a decisão em relação à abordagem da A1.** Com o mesmo otimizador, trocar a média histórica pela previsão
do Ridge levou o Sharpe de −0,12 para 0,20 e a riqueza final de R\$ 0,31 para R\$ 0,72. O ganho se mantém para os três valores
de regularização testados (Sharpe do ML entre 0,20 e 0,27, sempre acima da média histórica).

**2. Mas o ganho não vem de poder preditivo.** As duas estimativas erram quase igual: RMSE 0,0097 (ML) contra 0,0100,
acerto de direção de 52% contra 54% (praticamente cara ou coroa) e IC médio de −0,025 contra +0,022, ambos indistinguíveis de zero.
Em 30 dias, retornos de criptomoedas são dominados por ruído. A diferença operacional entre as duas é explicada pelo
**que cada uma errou na hora errada**: em janeiro de 2022 a média histórica apontava para os vencedores de 2021 (SOL, AVAX,
NEAR, DOGE), que foram os que mais caíram na crise de 2022.

**3. O controle sem previsão nenhuma vence tudo.** O mesmo otimizador recebendo só a covariância (mínima variância com
$K=5$) obteve Sharpe 0,80, +35% ao ano, o menor drawdown entre as carteiras otimizadas (−67,5%) e R\$ 2,46 de riqueza final,
superando inclusive o BTC (Sharpe 0,68) e o 1/N (0,34). Em média ele aloca cerca de 28% em BTC, 25% em BNB e 11% em ETH:
BTC e ETH são os dois ativos de menor volatilidade do universo, e o BNB entra pela baixa correlação com os demais.

**4. Consequência para a otimização.** A solução "ótima" da A1 é ótima para os parâmetros estimados, não para o futuro.
Como o modelo de média-variância maximiza onde $\mu$ é mais alto, ele amplifica justamente o erro de estimação de $\mu$
(o "maximizador de erros" de Michaud, 1989). A covariância é muito mais estável e estimável do que o retorno esperado
(Merton, 1980), e é dela que vem o valor do otimizador neste problema. Na prática: (a) a decisão operacional mais robusta é
otimizar o risco com as restrições de cardinalidade e limites; (b) um modelo de ML só deve substituir $\mu$ se mostrar IC
positivo e estável fora da amostra, o que não ocorreu aqui; (c) avaliar previsões só pelo erro (RMSE) esconderia isso:
o que importa para a carteira é a ordenação dos ativos (IC) e o efeito na decisão.

**Limitações:** um único período (2022 a 2024, com um forte ciclo de baixa e alta); universo escolhido em 2024 (viés de
sobrevivência: ativos que deixaram de ser líquidos não estão na base); otimizador heurístico no backtest (gap médio de
cerca de 1% ao ótimo, Seção 5); custo de transação simplificado.

---
## 5. Etapa 5 · Análise comparativa: solução exata x metaheurística (`FR-08`)

**Instância de referência:** modelo MINLP da A1 ($N=19$ ativos, $K=5$, pesos em $[5\%, 40\%]$, $\lambda=1$, retorno-alvo 0,10% ao dia).

| Método | Papel na comparação |
|---|---|
| **Couenne** (branch-and-bound espacial) | Solução **ótima exata** do MINLP, com prova de otimalidade |
| **Ipopt** na relaxação contínua (sem binárias, sem peso mínimo) | **Limitante inferior**: nenhuma solução viável pode ser melhor |
| **Algoritmo Genético** (10 sementes) | Metaheurística; qualquer solução viável é um **limitante superior** |
| **CBC** (modelo CVaR MILP da A1) | Outro modelo; entra só na comparação de recursos |

**Como medimos os recursos.** Os solvers rodam como executáveis separados, invisíveis para o alocador do Python. Por isso
`profile_call` mede: tempo de parede; tempo de CPU do processo Python **mais** o dos processos filhos (solver); pico de memória
residente (RSS) do processo do solver, amostrado a cada 5 ms; e o pico de memória alocada pelo Python (construção do modelo
Pyomo, AG) com `tracemalloc`, em uma execução separada para não distorcer o tempo.

In [ ]:
# FR-08: exato x limitante x metaheurística, com CPU e RAM (≈ 30 a 60 s)
from portfolio_opt.evaluation import run_exact_vs_heuristic
ev = run_exact_vs_heuristic(ret_matrix, cov_matrix, mean_returns, PARAMS, seeds=SEMENTES_AG, time_limit=LIMITE_TEMPO_S)
REF = ev.exact.objective_value

cols = ["Método", "Objetivo", "Gap vs Ótimo (%)", "Tempo de Parede (s)", "Tempo de CPU (s)", "Uso de CPU (%)",
        "RAM Solver Externo (MB)", "Pico Heap Python (MB)", "Observação"]
display(ev.resources[cols].set_index("Método").style.format(
    {"Objetivo": "{:.7f}", "Gap vs Ótimo (%)": "{:.4f}", "Tempo de Parede (s)": "{:.2f}", "Tempo de CPU (s)": "{:.2f}",
     "Uso de CPU (%)": "{:.0f}", "RAM Solver Externo (MB)": "{:.1f}", "Pico Heap Python (MB)": "{:.2f}"}, na_rep="—"))
display(ev.ga_seeds.set_index("Semente").style.format(
    {"Objetivo": "{:.7f}", "Gap vs Ótimo (%)": "{:.3f}", "Tempo de Parede (s)": "{:.2f}", "Tempo de CPU (s)": "{:.2f}"}))

lb, ub = ev.lower_bound.objective_value, ev.ga_seeds["Objetivo"].min()
print(f"Ótimo provado (Couenne): {REF:.7f}  ativos: {ev.exact.weights}")
print(f"Intervalo certificado só com relaxação + AG: [{lb:.7f}, {ub:.7f}] "
      f"-> largura relativa {abs(ub - lb) / abs(lb) * 100:.2f}%")

In [ ]:
# FR-08, FR-09: recursos e convergência
from portfolio_opt.dashboard import fig_resources, fig_convergence_gap, fig_convergence_diversity
from portfolio_opt.evaluation import convergence_table, generations_to_gap
fig_resources(ev, "Tempo de Parede (s)", "Tempo de parede por método", "s").show()
fig_resources(ev, "RAM Solver Externo (MB)", "Pico de RAM do processo do solver", "MB").show()
fig_convergence_gap(ev).show()
fig_convergence_diversity(ev).show()

ct = convergence_table(ev.ga_runs[0], REF)
display(ct.loc[ct["gen"].isin([1, 5, 10, 20, 40, 60, 80]), ["gen", "min_fit", "mean_fit", "gap_best_pct", "gene_std", "unique_supports"]]
        .rename(columns={"gen": "Geração", "min_fit": "Melhor", "mean_fit": "Média", "gap_best_pct": "Gap melhor (%)",
                         "gene_std": "Desvio dos genes", "unique_supports": "Conjuntos distintos"}).set_index("Geração")
        .style.format({"Melhor": "{:.6f}", "Média": "{:.6f}", "Gap melhor (%)": "{:.2f}", "Desvio dos genes": "{:.3f}",
                       "Conjuntos distintos": "{:.0f}"}).set_caption("Convergência da semente 0"))
for thr in [5, 2, 1]:
    g = [generations_to_gap(r, REF, thr) for r in ev.ga_runs.values()]
    print(f"Gerações até gap ≤ {thr}%: mediana {np.nanmedian(g):.0f} · sementes que chegaram: {np.sum(~np.isnan(g))}/{len(g)}")

### 5.1 Exploração x explotação
No AG, a **mutação** (desvio $\sigma$ da perturbação gaussiana) espalha a busca (exploração/diversificação) e a **pressão de
seleção** (tamanho do torneio) concentra a reprodução nos melhores (explotação/intensificação). A grade abaixo varia os dois,
com 5 sementes por célula, e mede qualidade, robustez, velocidade e a diversidade que sobra na população final.

In [ ]:
# FR-08: grade sigma x torneio (≈ 20 a 40 s)
from portfolio_opt.evaluation import run_exploration_grid
from portfolio_opt.dashboard import fig_exploration_grid
grid = run_exploration_grid(cov_matrix, mean_returns, PARAMS, REF, sigmas=SIGMAS_MUTACAO,
                            tournament_sizes=TAMANHOS_TORNEIO, seeds=SEMENTES_GRADE)
display(grid.style.format(precision=2, na_rep="não atingiu"))
fig_exploration_grid(grid).show()

### 5.2 Sensibilidade
Variamos um parâmetro por vez, mantendo os demais da instância de referência, e resolvemos **cada** variação pelos dois
métodos: cardinalidade $K$, aversão ao risco $\lambda$ e tamanho do universo $N$ (os $N$ ativos de maior retorno médio,
para manter a meta de retorno viável). Limite de tempo do Couenne: 60 s.
O gap aqui tem sinal: positivo, o AG ficou pior que o Couenne; negativo, o AG achou solução melhor (só possível quando o
Couenne para no limite de tempo, coluna "Couenne Provou Ótimo" = False).

In [ ]:
# FR-08: sensibilidade em K, lambda e N (≈ 1,5 a 3 minutos)
from portfolio_opt.evaluation import run_sensitivity
from portfolio_opt.dashboard import fig_sensitivity_gap, fig_sensitivity_times, fig_frontier
sens = run_sensitivity(cov_matrix, mean_returns, PARAMS, k_values=K_VALORES, lambda_values=LAMBDA_VALORES,
                       n_values=N_VALORES, time_limit=LIMITE_TEMPO_SENSIBILIDADE_S)
fmt = {"Objetivo Couenne": "{:.6f}", "Objetivo AG": "{:.6f}", "Gap AG (%)": "{:+.2f}", "Tempo Couenne (s)": "{:.2f}",
       "Tempo AG (s)": "{:.2f}", "Retorno Anual Ótimo": "{:.0%}", "Volatilidade Anual Ótima": "{:.0%}"}
for nome, tab in sens.items():
    display(tab.set_index(tab.columns[0]).style.format(fmt, na_rep="—").set_caption(f"Sensibilidade: {nome}"))
fig_sensitivity_gap(sens).show()
fig_sensitivity_times(sens["cardinalidade"], "K", "Tempo vs cardinalidade K").show()
fig_sensitivity_times(sens["num_ativos"], "N", "Tempo vs número de ativos N").show()
fig_frontier(sens).show()

### 5.3 Discussão crítica: solução exata x metaheurística
*Objetivos, gaps e ativos citados abaixo se reproduzem em qualquer execução. Tempos dependem da máquina e da carga dela: por isso citamos as faixas observadas em duas execuções completas (as tabelas acima trazem os da execução atual) e tiramos conclusões apenas das tendências, que se repetiram em todas.*

**a) Qualidade: ótimo, limitantes e gap.** O Couenne prova o ótimo do MINLP ($-0{,}0033932$; AVAX 5%, BNB 21%, DOGE 28%,
SOL 40%, XRP 5%). A relaxação contínua resolvida pelo Ipopt dá $-0{,}0033934$, um limitante inferior apenas 0,005% abaixo:
a relaxação é quase justa porque o ótimo contínuo já concentra o peso nos mesmos cinco ativos (só o XRP fica abaixo
dos 5% mínimos). O AG encontrou o melhor resultado em 0,27% do ótimo e, na média das 10 sementes, 0,96% ± 1,02%
(pior caso 3,3%). Todas as sementes acertaram **o mesmo conjunto de ativos** do ótimo: o erro do AG está só no ajuste fino
dos pesos, não na escolha combinatória. Isso sugere uma estratégia híbrida natural: AG para escolher os ativos e um solver
contínuo (Ipopt/QP) para os pesos. **Ponto metodológico:** mesmo sem o Couenne, relaxação e AG juntos cercam o ótimo em
um intervalo de cerca de 0,3% de largura. Em instâncias grandes, em que o exato não termina, esse par de limitantes é o que
permite afirmar a qualidade da heurística.

**b) Tempo, CPU e RAM.** Na instância de referência o Couenne levou 2 a 3 s, contra 1 a 2 s por execução do AG e menos de
1 s da relaxação. A comparação justa, porém, é com as 10 sementes que usamos para ter confiança no AG (10 a 20 s):
**aqui o exato é competitivo**, porque $N=19$ é pequeno. O uso de CPU acima de 100% no Couenne e no CBC indica trabalho em paralelo: o processo
do solver soma-se ao Python, e as bibliotecas numéricas podem usar mais de uma *thread*. O AG é puramente sequencial (≈ 100% de
um núcleo). A memória não é gargalo: o processo do Couenne atingiu ≈ 16 MB, o Ipopt ≈ 8 MB, e o AG aloca menos de 0,5 MB no Python.
O modelo mais "pesado" é o CVaR (CBC, ≈ 33 MB no solver e 4,5 MB no Python), porque tem uma restrição por cenário
(1.459 dias × 19 ativos). Ou seja, a memória escala com $T \times N$ nos modelos por cenários e com $N^2$ nos de covariância.

**c) Convergência.** O AG sai de um gap de ≈ 19% na 1ª geração para ≈ 7% na 10ª, ≈ 4% na 40ª e < 1% na 80ª: queda rápida no
início e lenta no final, típica da fase de refinamento. A diversidade colapsa cedo: o número de conjuntos de ativos distintos
na população cai de ≈ 94 para ≈ 20 em 20 gerações. A partir daí a busca é quase toda explotação, e o ganho restante vem de
pequenos ajustes de peso. O exato não tem "curva" comparável: ele só termina quando o limitante inferior encontra a melhor
solução, e é essa prova que custa tempo.

**d) Exploração x explotação.** A grade mostra o equilíbrio na prática:
- **$\sigma = 0{,}05$ (exploração fraca):** convergência prematura. A diversidade final é a menor da grade, o gap médio fica
  em 2,7–3,1% e, com torneio 10, nenhuma semente chega a 2%.
- **$\sigma = 0{,}6$ + torneio 10 (exploração e pressão excessivas):** pior célula (≈ 5%). A mutação destrói o que a seleção
  concentra.
- **Melhores resultados:** $\sigma = 0{,}6$ com torneio 2 (exploração forte com seleção branda) teve o menor gap médio (1,9%).
  $\sigma = 0{,}2$ com torneios 5 e 10 produziu os melhores indivíduos (0,18%) e chegou a 2% mais rápido (41–45 gerações),
  mas com maior variância entre sementes.

Mais pressão de seleção acelera a convergência mas aumenta o risco de ficar preso. Mais mutação protege contra isso, mas
atrasa o refinamento. O ponto de operação $\sigma = 0{,}2$, torneio 3 (o padrão) fica nessa região intermediária.

**e) Sensibilidade.**
- **Cardinalidade $K$:** o tempo do Couenne cresce com $K$ (≈ 1 s com $K=3$ e de 3 a 10 s com $K=10$), pois há mais
  combinações viáveis para o *branch-and-bound* descartar. O do AG não depende de $K$: o custo por geração é o mesmo. O gap do AG, porém, **cresce com
  $K$**: 0% ($K=3$), 0,8%, 1,7%, 3,0% e 9,7% ($K=10$). A causa é estrutural: o operador de reparo sempre ativa **exatamente**
  $K$ ativos com pelo menos 5% cada, mas o ótimo usa só 5 ativos mesmo quando $K=10$. A representação impede o AG de
  alcançar o ótimo. **$K=2$ é inviável** ($2 \times 40\% < 100\%$): o Couenne prova a inviabilidade em menos de 1 s, enquanto
  o AG devolve uma carteira que **viola** o peso máximo sem avisar. O pacote agora sinaliza isso (`is_feasible`). É a
  diferença mais importante entre os paradigmas: o exato prova ótimo e inviabilidade, e a heurística não prova nada.
- **Aversão ao risco $\lambda$:** aqui o exato é que sofre. Com $\lambda = 0{,}25$ (objetivo dominado pela variância) o Couenne
  **atingiu o limite de 60 s sem provar o ótimo**; com $\lambda=0{,}5$ levou de 9 a 28 s, e com $\lambda \geq 2$, cerca de 1 s.
  Quando a parte quadrática domina, as relaxações convexas que o Couenne usa para podar ramos ficam mais fracas. Com o
  tempo esgotado, o Couenne entrega apenas a melhor solução que tinha, sem garantia. Em uma das execuções essa solução era
  pior que a do AG (gap negativo na tabela): **sob limite de tempo, o exato perde a garantia e pode perder até para a
  heurística**, e o resultado passa a depender da velocidade da máquina. O tempo do AG, de novo, não depende de $\lambda$. Com $\lambda=4$ o gap do AG sobe para 8,6% pelo mesmo motivo de representação: o ótimo tem
  3 ativos e o AG é forçado a usar 5.
- **Tamanho do universo $N$:** o tempo do Couenne cresce com $N$ (menos de 1 s com $N=5$ e ≈ 2 s com $N=19$), refletindo o
  crescimento combinatório $\binom{N}{K}$. O AG é praticamente insensível a $N$ nesta faixa.

**f) Síntese: quando usar cada um.**

| Critério | Exato (Couenne) | Metaheurística (AG) |
|---|---|---|
| Garantia | Prova ótimo e inviabilidade | Nenhuma; precisa de limitantes externos |
| Tempo nesta instância | 2 a 3 s, mas imprevisível (60 s+ com $\lambda$ baixo) | 1 a 2 s por semente, sem depender de $K$, $\lambda$ ou $N$ |
| Escala com $N$ e $K$ | Cresce (explosão combinatória) | Praticamente constante |
| Qualidade | Ótimo | ≈ 1% em média; piora quando a representação não cobre o ótimo |
| Memória | Baixa (MBs) | Muito baixa |

Para este problema ($N=19$), **o exato é a escolha certa**: termina em segundos e dá garantia. O AG se justifica quando
$N$ cresce (centenas de ativos), quando o modelo ganha não-linearidades que o solver não trata, ou dentro de um backtest com
centenas de reotimizações, como na Etapa 4, em que estabilidade de tempo importa mais que o último 1%. Nesses casos ele
deve vir acompanhado de um limitante inferior (relaxação) para medir o gap e de uma checagem explícita de viabilidade.
Por fim, a Etapa 4 relativiza tudo isso: um gap de 1% na função objetivo é pequeno perto do erro de estimar $\mu$.
Na prática, a precisão do otimizador importa menos que a qualidade dos parâmetros que ele recebe.

---
## 6. Etapa 6 · Painel interativo (`FR-09`)
O painel reúne os objetivos do projeto (qualidade da solução, custo computacional, desempenho fora da amostra) em um único
arquivo HTML interativo (Plotly), salvo em `reports/dashboard_a2.html`. Ele abre direto no navegador (precisa de internet
apenas para carregar a biblioteca Plotly) e acompanha o tema claro/escuro do sistema. É o material de apoio da apresentação.

In [ ]:
# FR-09: geração do painel HTML
from portfolio_opt.dashboard import build_dashboard_html, build_figures, _kpis
figs = build_figures(wf, ev, grid, sens)
caminho = build_dashboard_html(wf, ev, grid, sens, "reports/dashboard_a2.html", figures=figs)
print(f"Painel salvo em: {caminho.resolve()}")
display(pd.DataFrame(_kpis(wf, ev)).rename(columns={"label": "Indicador", "value": "Valor", "note": "Contexto"}).set_index("Indicador"))
if IS_COLAB:
    print("Para baixar: painel lateral 'Arquivos' → reports/dashboard_a2.html → Download "
          "(ou execute: from google.colab import files; files.download(str(caminho)))")